In [5]:
import pandas as pd
import os
import requests

# Load datasets
btc = pd.read_csv("data/silver/btc_daily_clean.csv", parse_dates=["date"])
fng = pd.read_csv("data/silver/fng_daily_clean.csv", parse_dates=["date"])

# Merge on date
merged = pd.merge(btc, fng, on="date", how="inner")

# Feature engineering
merged["btc_return_pct"] = merged["btc_close"].pct_change()
merged["fear"] = merged["fng_value"] < 47

# -------------------------------
# Holiday API (Nager.Date)
# -------------------------------

years = merged["date"].dt.year.unique()
holiday_list = []

for year in years:
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/CA"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        holiday_list.extend(data)
    else:
        print(f"Failed to fetch holidays for {year}")

# Convert to DataFrame
holidays_df = pd.DataFrame(holiday_list)

# Keep only needed columns
holidays_df = holidays_df[["date", "localName"]]
holidays_df["date"] = pd.to_datetime(holidays_df["date"])

holidays_df = holidays_df.rename(columns={"localName": "holiday_name"})

# -------------------------------
# Deduplicate holidays (collapse multiple provincial holidays on same date)
# -------------------------------
holidays_df = (
    holidays_df
    .groupby("date")["holiday_name"]
    .apply(lambda x: " / ".join(sorted(set(x))))
    .reset_index()
)

# -------------------------------
# Merge holidays into dataset
# -------------------------------
merged = pd.merge(merged, holidays_df, on="date", how="left")
# Create boolean flag
merged["is_holiday"] = merged["holiday_name"].notna()

# Fill non-holidays
merged["holiday_name"] = merged["holiday_name"].fillna("None")
holiday_dates = merged.loc[merged["is_holiday"], "date"]

merged["days_to_holiday"] = merged["date"].apply(
    lambda d: min(abs((d - h).days) for h in holiday_dates) if len(holiday_dates) > 0 else None
)

# -------------------------------
# Optional: weekend flag
# -------------------------------
merged["is_weekend"] = merged["date"].dt.weekday >= 5

print(merged.head())
print(f"Rows: {len(merged)}")

# Ensure gold folder exists
os.makedirs("data/gold", exist_ok=True)

# Save WITHOUT overwriting original
output_path = "data/gold/btc_fng_with_holidays_api.csv"
merged.to_csv(output_path, index=False)

print(f"[SUCCESS] Saved to {output_path}")

        date  btc_close   btc_volume  gain_loss   gain  fng_value  \
0 2025-04-02   82516.29  39931.45700       0.00  False         44   
1 2025-04-03   83213.09  27337.84135     696.80   True         25   
2 2025-04-04   83889.87  32915.53976     676.78   True         28   
3 2025-04-05   83537.99   9360.40468    -351.88  False         30   
4 2025-04-06   78430.00  27942.71436   -5107.99  False         34   

  fng_classification  btc_return_pct  fear holiday_name  is_holiday  \
0               Fear             NaN  True         None       False   
1       Extreme Fear        0.008444  True         None       False   
2               Fear        0.008133  True         None       False   
3               Fear       -0.004195  True         None       False   
4               Fear       -0.061146  True         None       False   

   days_to_holiday  is_weekend  
0               16       False  
1               15       False  
2               14       False  
3               13        